In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'stage_0_0_setup_orchestrator.py').exists():
    candidate = REPO_ROOT / 'CoChem-CORE'
    if (candidate / 'stage_0_0_setup_orchestrator.py').exists():
        REPO_ROOT = candidate

cmd = [sys.executable, str(REPO_ROOT / 'stage_0_0_setup_orchestrator.py')]
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, cwd=str(REPO_ROOT), check=False)
if result.returncode != 0:
    raise RuntimeError(f'stage_0_0_setup_orchestrator.py failed with exit code {result.returncode}')
print('CoChem-CORE setup completed.')

# CoChem-MInt Molecule Intake
Run the next cell to load the molecule intake widget. Add your `.xyz` files to `CoChem_Artifacts/Input_Files`, then use the widget controls to scan/build/watch.

In [ ]:
import importlib.util

mint_script = REPO_ROOT / 'intake' / 'CoChem-MInt.py'
if not mint_script.exists():
    raise FileNotFoundError(f'Missing CoChem-MInt script: {mint_script}')

spec = importlib.util.spec_from_file_location('cochem_mint', str(mint_script))
mint_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mint_module)

mint_ui = mint_module.CoChemMIntUI()
mint_ui.display()

# Clone CoChem-TOPOS and CoChem-TORQ
This cell attempts to clone both repositories next to `CoChem-CORE` in the same parent workspace directory.

In [ ]:
workspace_root = REPO_ROOT.parent
repos = {
    'CoChem-TOPOS': 'https://github.com/CoChem/CoChem-TOPOS',
    'CoChem-TORQ': 'https://github.com/CoChem/CoChem-TORQ',
}

for name, url in repos.items():
    target = workspace_root / name
    if target.exists():
        print(f'Already exists: {target}')
        continue

    probe = subprocess.run(['git', 'ls-remote', '--heads', url], capture_output=True, text=True, check=False)
    if probe.returncode != 0:
        alt_url = f'{url}.git'
        probe_alt = subprocess.run(['git', 'ls-remote', '--heads', alt_url], capture_output=True, text=True, check=False)
        if probe_alt.returncode != 0:
            print(f'Clone skipped for {name}: repository not reachable from this environment.')
            print((probe.stderr or probe_alt.stderr or '').strip())
            continue
        url = alt_url

    clone = subprocess.run(['git', 'clone', '--depth', '1', url, str(target)], check=False)
    if clone.returncode == 0:
        print(f'Cloned {name} -> {target}')
    else:
        print(f'Clone failed for {name} (exit {clone.returncode})')

# Configure Manifest and Build Silos
This cell writes a manifest selecting `CoChem-CORE`, `CoChem-TOPOS`, and `CoChem-TORQ`, then runs Phase 4 and Phase 5 to ensure silos are prepared.

In [ ]:
import json

setup_dir = REPO_ROOT / 'cochem_setup'
setup_dir.mkdir(parents=True, exist_ok=True)
manifest_path = setup_dir / 'cochem_deployment_manifest.json'

manifest = {
    'version': '2026.2',
    'deployment_target': 'Local DevContainer',
    'modules': ['CoChem-CORE', 'CoChem-TOPOS', 'CoChem-TORQ'],
    'selected_modules': {
        'CoChem-CORE': 'https://github.com/CoChem/CoChem-CORE',
        'CoChem-TOPOS': 'https://github.com/CoChem/CoChem-TOPOS',
        'CoChem-TORQ': 'https://github.com/CoChem/CoChem-TORQ'
    },
    'host_orca_path': ''
}

manifest_path.write_text(json.dumps(manifest, indent=4), encoding='utf-8')
print(f'Manifest written: {manifest_path}')

for phase in ['cochem_setup_phase_4.py', 'cochem_setup_phase_5.py']:
    phase_path = setup_dir / phase
    cmd = [sys.executable, str(phase_path)]
    print('Running:', ' '.join(cmd))
    rc = subprocess.run(cmd, cwd=str(REPO_ROOT), check=False).returncode
    if rc != 0:
        raise RuntimeError(f'{phase} failed with exit code {rc}')

print('Silo setup phases completed.')

# Run Setup for Cloned Modules
This cell runs each cloned module orchestrator if present to complete module-specific setup.

In [ ]:
module_dirs = [REPO_ROOT.parent / 'CoChem-TOPOS', REPO_ROOT.parent / 'CoChem-TORQ']
for module_dir in module_dirs:
    if not module_dir.exists():
        print(f'Skipping missing module directory: {module_dir}')
        continue

    orchestrator = module_dir / 'stage_0_0_setup_orchestrator.py'
    if not orchestrator.exists():
        print(f'No setup orchestrator found in {module_dir.name}; skipping module setup.')
        continue

    cmd = [sys.executable, str(orchestrator)]
    print(f"Running {module_dir.name} setup: {' '.join(cmd)}")
    rc = subprocess.run(cmd, cwd=str(module_dir), check=False).returncode
    if rc != 0:
        raise RuntimeError(f'{module_dir.name} setup failed with exit code {rc}')

print('TOPOS/TORQ module setup pass complete.')

# Optional: Show Recent Logs

In [ ]:
artifact_dir = Path(os.environ.get('COCHEM_ARTIFACT_DIR', str(Path.home() / 'CoChem_Artifacts')))
log_dir = artifact_dir / 'Logs'
print(f'Log directory: {log_dir}')

if not log_dir.exists():
    print('No logs found yet.')
else:
    for path in sorted(log_dir.glob('*.log')):
        print(f'\n=== {path.name} ===')
        lines = path.read_text(errors='replace').splitlines()
        print('\n'.join(lines[-40:]) if lines else '<empty>')

In [ ]:
from IPython.display import display

dashboard_script = REPO_ROOT / 'interfaces' / 'cochem_unity_installer_dashboard.py'
if not dashboard_script.exists():
    raise FileNotFoundError(f'Missing UNITY dashboard script: {dashboard_script}')

dashboard_spec = importlib.util.spec_from_file_location('cochem_unity_installer_dashboard', str(dashboard_script))
dashboard_module = importlib.util.module_from_spec(dashboard_spec)
dashboard_spec.loader.exec_module(dashboard_module)

installer = dashboard_module.UnityInstallerGUI()
display(installer.main_ui)